### BUSI de-duplicationPerceptual hashing used to clean the BUSI dataset before splitting, as described in Section 2.3of the manuscript. 64-bit pHash (8x8 DCT), Hamming distance threshold of 4.Should give 152 removed and 628 kept (340 benign, 193 malignant, 95 normal).BUSI is from Al-Dhabyani et al., Data in Brief 28 (2020) 104863. Not redistributed here,download it separately and set the path below.

In [ ]:
!pip install -q imagehash

In [ ]:
import osimport globfrom collections import Counterimport pandas as pdimport imagehashfrom PIL import Imagebusi_root = "Dataset_BUSI_with_GT"   # change thisthresh = 4

In [ ]:
# masks are stored next to the images as *_mask.png, drop themfiles = sorted(f for f in glob.glob(busi_root + "/*/*.png") if "_mask" not in f)len(files)

In [ ]:
hashes = {}for f in files:    img = Image.open(f).convert("L")    hashes[f] = imagehash.phash(img, hash_size=8)

In [ ]:
# go through in order, keep the first of each group and drop anything close to itkept = []dropped = []for f in files:    dup_of = None    for k in kept:        if hashes[f] - hashes[k] <= thresh:            dup_of = k            break    if dup_of is None:        kept.append(f)    else:        dropped.append([os.path.relpath(f, busi_root),                        os.path.relpath(dup_of, busi_root),                        hashes[f] - hashes[dup_of]])print("dropped:", len(dropped))print("kept:", len(kept))

In [ ]:
counts = Counter(f.split("/")[-2] for f in kept)print(counts)print("benign + malignant =", counts["benign"] + counts["malignant"])

In [ ]:
df = pd.DataFrame(dropped, columns=["removed_image", "kept_representative", "hamming_distance"])df = df.sort_values("removed_image").reset_index(drop=True)df.to_csv("busi_near_duplicates_removed.csv", index=False)df.head()